# Interactive Narrative & Logic Workshop
### Project Overview
This project is an AI-powered creative writing platform developed for the "AMD ROCm x LLM Application Development Challenge." It functions as both a co-authoring assistant and a rigorous "narrative auditor."
- **Core Technology**: Optimized for AMD GPU acceleration using the `qwen3-coder:30b` model with a 32k context window.
- **Key Features**:
  1. **Interactive Co-creation**: Multi-turn story generation across various genres (Sci-Fi, Mystery, Cyberpunk, etc.).
  2. **Logic Consistency Auditing**: Leverages the LLM as a "Logic Expert" to automatically detect plot gaps and narrative contradictions.
  3. **Structural Analysis**: Performs "Three-Act Structure" diagnostics and scoring based on literary theory to enhance story quality.
  4. **Engineering Optimization**: Implements dynamic 32k context truncation strategies and an asynchronous, non-blocking UI using `ipywidgets`.

In [ ]:
# Check AMD GPU availability
import torch
print(f'ROCm available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import requests
import json
import threading
import time
import re

# === 1. Global Configuration & State ===
API_URL = "http://open-webui-ollama.open-webui:11434/api/chat"
MODEL_NAME = "qwen3-coder:30b"
MAX_CHARS_LIMIT = 90000 

story_history = []  # {role, content, type: 'story'/'analysis', story_text, req_text}
stop_event = threading.Event()
is_generating = False # Flag for LLM response status

# === 2. Enhanced UI Styles & Auto-scroll Script ===
display(HTML("""
<style>
    .story-canvas {
        height: 580px; overflow-y: auto; padding: 25px;
        background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px;
        line-height: 1.8; font-size: 15px; margin-bottom: 15px;
        scroll-behavior: smooth;
    }
    .msg-user-story { border-left: 5px solid #10b981; background: #f0fdf4; padding: 12px; margin-top: 15px; border-radius: 8px; font-weight: 500; }
    .msg-user-req { font-size: 13px; color: #64748b; padding-left: 15px; margin-bottom: 10px; font-style: italic; }
    .msg-bot-story { padding: 5px 0 25px 0; color: #1e293b; white-space: pre-wrap; transition: opacity 0.5s; }
    
    /* Thinking Placeholder Style */
    .thinking-block { 
        padding: 15px; color: #94a3b8; font-style: italic; 
        background: #f8fafc; border-radius: 8px; border: 1px dashed #e2e8f0;
        margin: 10px 0; animation: pulse 1.5s infinite;
    }
    @keyframes pulse { 0% { opacity: 0.6; } 50% { opacity: 1; } 100% { opacity: 0.6; } }

    /* Analysis Report Style */
    .msg-analysis {
        background: #f8fafc; border: 1px solid #e2e8f0; border-top: 4px solid #3b82f6;
        border-radius: 8px; padding: 20px; margin: 25px 0;
    }
    .msg-analysis h2 { color: #1e3a8a; font-size: 1.2em; border-bottom: 1px solid #cbd5e1; padding-bottom: 5px; margin-top: 15px; }
    .msg-analysis h3 { color: #2563eb; font-size: 1.1em; margin-top: 12px; }
    .score-badge { background: #3b82f6; color: white; padding: 1px 10px; border-radius: 10px; font-weight: bold; }

    .typing-cursor {
        display: inline-block; width: 8px; height: 18px; background: #3b82f6; 
        animation: blink 0.8s infinite; margin-left: 5px; vertical-align: middle;
    }
    @keyframes blink { 0%, 100% { opacity: 1; } 50% { opacity: 0; } }
</style>

<script>
(function() {
    const autoScroll = () => {
        const el = document.querySelector('.story-canvas');
        if (!el) return;
        const threshold = 100;
        const isAtBottom = el.scrollHeight - el.scrollTop - el.clientHeight < threshold;
        
        const observer = new MutationObserver(() => {
            if (isAtBottom) { el.scrollTop = el.scrollHeight; }
        });
        observer.observe(el, { childList: true, subtree: true });
    };

    const init = () => {
        if (document.querySelector('.story-canvas')) autoScroll();
        else setTimeout(init, 500);
    };
    init();
})();
</script>
"""))

# === 3. Core Utility Functions: Parsing & Context Management ===

def format_pretty_html(text, mode="story"):
    if mode == "story":
        return text.replace("\n", "<br/>")
    
    # Strict Markdown-to-HTML parsing for analysis
    text = re.sub(r'^## (.*?)$', r'<h2>\1</h2>', text, flags=re.M)
    text = re.sub(r'^### (.*?)$', r'<h3>\1</h3>', text, flags=re.M)
    text = re.sub(r'(\d+/100)', r'<span class="score-badge">\1</span>', text)
    text = re.sub(r'\*\*(.*?)\*\*', r'<strong>\1</strong>', text)
    
    lines = text.split('\n')
    output = []
    in_list = False
    for line in lines:
        line = line.strip()
        if not line: continue
        if line.startswith(('- ', '* ')):
            if not in_list: output.append('<ul>'); in_list = True
            output.append(f'<li>{line[2:]}</li>')
        else:
            if in_list: output.append('</ul>'); in_list = False
            if not line.startswith('<h'): output.append(f'<p>{line}</p>')
            else: output.append(line)
    if in_list: output.append('</ul>')
    return "".join(output)

def apply_context_management(history):
    total_len = sum(len(m.get('content', '') or m.get('story_text', '')) for m in history)
    if total_len < MAX_CHARS_LIMIT:
        return history
    
    # Keep System Prompt, the opening message, and the last 8 rounds
    head = history[:2]
    tail = history[-8:]
    return head + tail

# === 4. UI Component Construction ===

title_html = widgets.HTML("<h2 style='color:#1e293b; margin-left:10px;'>📖 Narrative AI Workshop <span style='font-size:12px;color:#94a3b8;'></span></h2>")
genre_drop = widgets.Dropdown(options=['Sci-Fi', 'Mystery', 'Fantasy', 'Wuxia', 'Cyberpunk'], value='Sci-Fi', description='📚 Genre')
len_drop = widgets.Dropdown(options=[('Short (250 chars)', 250), ('Medium (550 chars)', 550), ('Detailed (850 chars)', 850)], value=550, description='📏 Length')
controls = widgets.HBox([genre_drop, len_drop])

story_display = widgets.HTML(value="<div class='story-canvas'>Waiting for the spark of inspiration...</div>")
story_input = widgets.Textarea(placeholder='Continue the story here...', layout={'width':'99%', 'height':'180px'})
req_input = widgets.Textarea(placeholder='(Optional) Enter plot instructions, e.g., "The villain suddenly counter-attacks"...', layout={'width':'99%', 'height':'45px'})

auto_btn = widgets.Button(description='✍️ Auto Continue', button_style='primary', layout={'width':'49%', 'height':'40px'})
guided_btn = widgets.Button(description='🎯 Guided Continue', button_style='success', layout={'width':'49%', 'height':'40px'})
consistency_btn = widgets.Button(description='🧐 Consistency Check', button_style='info', layout={'width':'32%'})
structure_btn = widgets.Button(description='📊 Structural Analysis', button_style='info', layout={'width':'32%'})
stop_btn = widgets.Button(description='⏸️ Stop', button_style='warning', disabled=True, layout={'width':'15%'})
undo_btn = widgets.Button(description='↩️ Undo', button_style='danger', layout={'width':'15%'})

app_ui = widgets.VBox([
    title_html, controls, story_display, 
    widgets.VBox([story_input, req_input], layout={'border':'1px solid #e2e8f0','padding':'10px','border_radius':'12px','background':'#fff'}),
    widgets.HBox([auto_btn, guided_btn]),
    widgets.HBox([consistency_btn, structure_btn, stop_btn, undo_btn])
])

def update_ui_canvas():
    """Real-time rendering of history with 'Thinking' placeholder"""
    html = "<div class='story-canvas'>"
    for m in story_history:
        if m.get("type") == "analysis":
            html += f"<div class='msg-analysis'>{format_pretty_html(m['content'], 'analysis')}</div>"
        else:
            if m["role"] == "user":
                if m.get('story_text'): html += f"<div class='msg-user-story'>[Input] {m['story_text']}</div>"
                if m.get('req_text'): html += f"<div class='msg-user-req'>Instruction: {m['req_text']}</div>"
            else:
                html += f"<div class='msg-bot-story'>{format_pretty_html(m['content'])}</div>"
    
    if is_generating:
        html += "<div class='thinking-block'>Literary inspiration flowing...<span class='typing-cursor'></span></div>"
        
    html += "</div>"
    story_display.value = html

# === 5. Backend Logic ===

def llm_executor(chat_context, record_type):
    global is_generating
    try:
        response = requests.post(API_URL, json={
            "model": MODEL_NAME, 
            "messages": chat_context, 
            "stream": False,
            "options": {"num_ctx": 32768, "temperature": 0.8}
        }, timeout=150)
        
        result = response.json()
        if "message" in result:
            final_content = result["message"].get("content", "")
            if not stop_event.is_set():
                story_history.append({"role": "assistant", "content": final_content, "type": record_type})
    except Exception as e:
        story_history.append({"role": "assistant", "content": f"System Error: {str(e)}", "type": "analysis"})
    
    is_generating = False
    set_ui_state(False)
    update_ui_canvas()

def trigger_generation(mode="auto"):
    global is_generating
    s_val, r_val = story_input.value.strip(), req_input.value.strip()
    
    if not s_val:
        if not story_history:
            return
        else:
            # If input is empty but continue is triggered, inject a default prompt
            current_req = r_val if r_val else "Please continue the plot based on the previous context."
            story_history.append({
                "role": "user", 
                "type": "story", 
                "story_text": "[Trigger: Continue Plot]", 
                "req_text": current_req
            })
    else:
        story_history.append({
            "role": "user", 
            "type": "story", 
            "story_text": s_val, 
            "req_text": r_val if mode=="guided" else ""
        })
    story_input.value, req_input.value = "", ""
    
    is_generating = True
    stop_event.clear()
    set_ui_state(True)
    update_ui_canvas()

    safe_history = apply_context_management(story_history)
    
    # Story Creation System Prompt
    sys_prompt = f"You are a world-class {genre_drop.value} novelist. Please continue writing approximately {len_drop.value} words mimicking the previous tone. Ensure delicate emotions and strict plot logic."
    
    messages = [{"role": "system", "content": sys_prompt}]
    for m in safe_history:
        if m.get("type") == "story":
            content = m["content"] if m["role"]=="assistant" else f"Text: {m.get('story_text','')}\nRequirement: {m.get('req_text','')}"
            messages.append({"role": m["role"], "content": content})
            
    threading.Thread(target=llm_executor, args=(messages, "story")).start()

def trigger_analysis(a_type):
    global is_generating
    if not story_history: return
    
    is_generating = True
    stop_event.clear()
    set_ui_state(True)
    update_ui_canvas()

    pure_text = "\n".join([m.get("content", "") if m["role"]=="assistant" else m.get("story_text", "") for m in story_history if m.get("type")=="story"])
    
    if a_type == "consistency":
        prompt = f"As a senior logic audit expert, please verify the following plot. Do not output JSON. Report in the following format:\n## 1. Key Elements List\n## 2. Conflicts from Potential Logic Loopholes\n## 3. Consistency Score\n\nContent: {pure_text[-5000:]}"
    else:
        prompt = f"As a PhD in Literature, please perform a three-act structure analysis of the following work. Do not output JSON. Report in the following format:\n## Overall Score: [Score]/100\n## 1. Structural Diagnosis\n## 2. Pacing Evaluation\n## 3. Recommendations\n\nContent: {pure_text[:7000]}"

    threading.Thread(target=llm_executor, args=([{"role": "system", "content": "Rigorous literary analysis assistant."},{"role": "user", "content": prompt}], "analysis")).start()

def set_ui_state(gen):
    for b in [auto_btn, guided_btn, consistency_btn, structure_btn, undo_btn]: b.disabled = gen
    stop_btn.disabled = not gen

# === 6. Event Binding & Display ===
auto_btn.on_click(lambda b: trigger_generation("auto"))
guided_btn.on_click(lambda b: trigger_generation("guided"))
consistency_btn.on_click(lambda b: trigger_analysis("consistency"))
structure_btn.on_click(lambda b: trigger_analysis("structure"))
stop_btn.on_click(lambda b: stop_event.set())
undo_btn.on_click(lambda b: (story_history.pop() if story_history else None, update_ui_canvas()))

display(app_ui)